# Milestone 1 — Cora + label and 1-WL partitions

Load Cora, build the *label partition* (7 cells, $e_C = 0$ everywhere) and the **architecture-induced partition** $\Pi_\mathcal{A}(G, L)$ (Def 3.1) at depths $L \in \{1,2,3\}$, instantiated by 1-WL. Verify **Proposition 3.2** numerically: refinement *shrinks both* $H(Y\mid\Pi)$ *and* $\varepsilon^*_\Pi$ (binarised).

**Julia companion (optional).** [`julia-theory/notebooks/08_refinement_monotonicity.jl`](../../../julia-theory/notebooks/08_refinement_monotonicity.jl) is the reactive twin: split a cell with a slider; watch the bracket interval contract.

## Setup

Resolve `onboarding/projects/` on `sys.path`, attach the reflection log, and import the canonical references (used only for spot-checks; the student version derives its own implementations).

In [ ]:
import os, sys
from pathlib import Path
_here = Path(os.getcwd()).resolve()
for _p in [_here, *_here.parents]:
    if (_p / 'shared' / '__init__.py').exists() and _p.name == 'projects':
        _PROJECTS = _p
        break
else:
    raise RuntimeError('could not locate onboarding/projects/')
_REPO = _PROJECTS.parent.parent
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))

import numpy as np
import matplotlib
matplotlib.use('Agg')   # headless for CI
import matplotlib.pyplot as plt

from onboarding.projects import reflect
reflect.start(notebook='m1', store=_PROJECTS / '.reflection.jsonl')
print(f'[setup] projects root: {_PROJECTS}')


In [ ]:
from onboarding.projects.shared import (
    Partition, label_partition, wl_partition,
    hbin, hbin_inverse, upper, lower, slack, verify, bracket_of,
    sum_partition, mean_partition, max_partition,
)
print('shared modules imported')


In [ ]:
from torch_geometric.datasets import Planetoid
_CACHE = Path.home() / '.cache' / 'planetoid'
_CACHE.mkdir(parents=True, exist_ok=True)
_cora = Planetoid(root=str(_CACHE / 'Cora'), name='Cora')[0]
n = int(_cora.num_nodes)
m = int(_cora.num_edges)
K = int(_cora.y.max().item()) + 1
print(f'Cora: n={n}, m={m}, classes={K}, x.shape={tuple(_cora.x.shape)}')


## Label partition
7 cells, one per class, $e_C = 0$ by construction.

In [ ]:
P_lbl = label_partition(_cora)
print(f'label partition: {len(P_lbl.cells)} cells; e_C={P_lbl.e}')


In [ ]:
assert len(P_lbl.cells) == K, f'M1: label partition has {len(P_lbl.cells)} cells, expected {K}'
assert all(e == 0 for e in P_lbl.e), f'M1: label partition has nonzero e_C: {P_lbl.e}'
assert abs(sum(P_lbl.q) - 1.0) < 1e-12
print('[GATE OK] M1.label: 7 cells, all e_C=0, q sums to 1')


## 1-WL partitions at $L \in \{1, 2, 3\}$ — **Prop 3.2** numerical audit.

Prop 3.2 says: if $\Pi'$ refines $\Pi$ then $H(Y\mid\Pi') \le H(Y\mid\Pi)$ and (after binarisation) $\varepsilon^*_{\Pi'} \le \varepsilon^*_\Pi$. Each WL depth $L+1$ refines $L$, so both quantities must be **non-increasing in $L$**.

In [ ]:
P_wl = {L: wl_partition(_cora, depth=L) for L in (1, 2, 3)}
def H_eps(P):
    bin_e = np.array([min(e, 1-e) for e in P.e])
    q = np.array(P.q)
    return float(np.sum(q * np.array([hbin(e) for e in bin_e]))), float(np.sum(q * bin_e))
for L, P in P_wl.items():
    H_, e_ = H_eps(P)
    print(f'L={L}: {len(P.cells)} cells; H(Y|Π)={H_:.4f}; ε*={e_:.4f}; max e_C={max(P.e):.4f}')


In [ ]:
ncells = [len(P_wl[L].cells) for L in (1,2,3)]
assert ncells[0] <= ncells[1] <= ncells[2], f'M1: refinement not monotone in cell count: {ncells}'
# Prop 3.2 (real): H and ε* non-increasing under refinement (binarised).
Hs   = [H_eps(P_wl[L])[0] for L in (1,2,3)]
epss = [H_eps(P_wl[L])[1] for L in (1,2,3)]
assert Hs[0]   + 1e-12 >= Hs[1]   >= Hs[2]   - 1e-12, f'M1.Prop3.2: H not monotone non-increasing: {Hs}'
assert epss[0] + 1e-12 >= epss[1] >= epss[2] - 1e-12, f'M1.Prop3.2: ε* not monotone non-increasing: {epss}'
# Multi-class e_C ∈ [0, (K-1)/K]; binarisation min(e, 1-e) ∈ [0, 1/2] is what the bracket consumes.
for L, P in P_wl.items():
    assert all(0 <= e <= 1 - 1/K + 1e-12 for e in P.e), f'M1: bad e_C at L={L}'
    bin_e = [min(e, 1 - e) for e in P.e]
    assert all(0 <= e <= 0.5 + 1e-12 for e in bin_e), f'M1: binarised e_C out of [0,1/2] at L={L}'
    assert abs(sum(P.q) - 1.0) < 1e-12
print(f'[GATE OK] M1.wl + Prop 3.2: ncells={ncells} monotone; H={[round(h,4) for h in Hs]} ↓; ε*={[round(e,4) for e in epss]} ↓')


## Histogram of per-cell $e_C$ for $L=2$

In [ ]:
fig, ax = plt.subplots(figsize=(6,3.5))
ax.hist(P_wl[2].e, bins=30, weights=P_wl[2].q, color='C0', alpha=0.85)
ax.set_xlabel('per-cell Bayes error $e_C$'); ax.set_ylabel('mass $q_C$')
ax.set_title(f'M1 — 1-WL(L=2) per-cell error histogram ({len(P_wl[2].cells)} cells)')
_plots = _PROJECTS / 'capstone' / 'milestone1' / 'plots'; _plots.mkdir(exist_ok=True)
plt.tight_layout(); fig.savefig(_plots / 'm1_ehist.png', dpi=120); plt.show()


In [ ]:
reflect.log('M1.partitions', f'label=7×0; wl(L=2) has {len(P_wl[2].cells)} cells, max e_C={max(P_wl[2].e):.3f}', 'HIGH')


**End of M1.**